# Validación experimental de reconstrucción DSA

Copia presentable generada a partir de `unilat_bilat.ipynb`.

Esta versión ejecuta el bloque unilateral y deja visibles las tablas de métricas usadas para justificar el suavizado y el desplazamiento temporal. El bloque bilateral se conserva como referencia de código, pero no se recalcula completo en esta copia porque el registro disponible de 17 h hace que la ejecución completa sea lenta para revisión rápida.


In [1]:
import sys
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import struct


import funciones_aux as fau

import funciones_dsa as fun_dsa
import funciones_dsa_unilateral as fun_dsa_u
import funciones_dsa_bilateral as fun_dsa_b

import funciones_plot_dsa as fun_plot

from scipy.signal import welch
from matplotlib.colors import LinearSegmentedColormap, PowerNorm
from scipy.stats import pearsonr, spearmanr


In [2]:
# ============================================================
# Parámetros fijos del flujo de reconstrucción
# ============================================================

# El suavizado espectral se obtiene del campo SpSmooth del .spa.
# El shift temporal se fija por modo a partir de las pruebas realizadas
# en registros donde existía archivo .f_a de referencia.

SHIFT_UNILAT = 10   # segundos
SHIFT_BILAT = 6     # segundos

# 1. Unilateral

In [3]:

ruta_fa_unilat = "C:/Users/usuario/Downloads/data/data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.f_a"
ruta_spa_unilat = "C:/Users/usuario/Downloads/data/data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.spa"
ruta_ha_unilat = "C:/Users/usuario/Downloads/data/data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.h_a"
ruta_ta_unilat = "C:/Users/usuario/Downloads/data/data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.t_a"

archivo_r2a = r"C:/Users/usuario/Downloads/data/data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.r2a"


In [4]:
# este es para el bis vista que no tiene fa
"""
ruta_spa_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_antiguo/L03041419/L03041419/L03041419.spa"
ruta_ha_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_antiguo/L03041419/L03041419/L03041419.h_a"
ruta_ta_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_antiguo/L03041419/L03041419/L03041419.t_a"

archivo_r2a = r"C:/Users/usuario/TFG_BIS_GIS/data/data_bis_antiguo/L03041419/L03041419/L03041419.r2a"
"""

'\nruta_spa_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_antiguo/L03041419/L03041419/L03041419.spa"\nruta_ha_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_antiguo/L03041419/L03041419/L03041419.h_a"\nruta_ta_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_antiguo/L03041419/L03041419/L03041419.t_a"\n\narchivo_r2a = r"C:/Users/usuario/TFG_BIS_GIS/data/data_bis_antiguo/L03041419/L03041419/L03041419.r2a"\n'

## 1. 1.  Archivo Espectral .f_a

In [5]:
tiempo_fa_unilat, dsa_unilat = fau.cargar_fa_directo(ruta_fa_unilat, escalar_db=True)

print("Dimensiones de la matriz:", dsa_unilat.shape)
print("Frecuencias:", dsa_unilat.columns.min(), "a", dsa_unilat.columns.max(), "Hz")

Dimensiones de la matriz: (2119, 60)
Frecuencias: 0.5 a 30.0 Hz


## 1. 2. Archivo variables procesadas .spa

In [6]:
df_spa_raw = fau.procesar_spa(ruta_spa_unilat)
df_spa_unilat = fun_dsa_u.limpiar_spa_unilateral(df_spa_raw)

print("Dimensiones del archivo procesado:", df_spa_unilat.shape)
display(df_spa_unilat.head())

Dimensiones del archivo procesado: (2119, 13)


,Time,SpSmooth,LoFilter,SEF08,MEDFRQ08,SQI10,TOTPOW08,EMGLOW01,SR12,ST,DB13U01,ARTF2,BURST
0,2026-03-04 10:35:21,3,3,NaN,NaN,0.0,NaN,56.9,NaN,0.0,NaN,NaN,NaN
1,2026-03-04 10:35:22,3,3,NaN,NaN,0.0,NaN,54.1,NaN,0.0,NaN,2000000.0,NaN
2,2026-03-04 10:35:23,3,3,NaN,NaN,0.0,NaN,58.4,NaN,0.0,NaN,2000000.0,NaN
3,2026-03-04 10:35:24,3,3,NaN,NaN,0.0,NaN,57.6,NaN,0.0,NaN,88.0,NaN
4,2026-03-04 10:35:25,3,3,NaN,NaN,0.8,NaN,60.4,NaN,0.0,NaN,180.0,NaN


In [7]:
# Extraer SpSmooth del .spa y traducirlo a segundos

valor_spsmooth_unilat, suavizado_spsmooth_unilat, dist_spsmooth_unilat = (
    fau.extraer_spsmooth_segundos(
        df_spa_unilat,
        verbose=True
    )
)

SpSmooth codificado más frecuente: 3
Suavizado espectral equivalente: 30 s

Distribución de valores SpSmooth:


3    2119
Name: SpSmooth, dtype: int64

### timeline oficial del .spa

In [8]:
timeline_spa_unilat = fun_dsa_b.preparar_timeline_spa(
    df_spa=df_spa_unilat,
    columna_time="Time",
    resolver_duplicados="last",
    verbose=True
)

=== Timeline .spa ===
Inicio .spa: 2026-03-04 10:35:21
Fin .spa: 2026-03-04 11:10:39
N segundos .spa: 2119


### ajustar .f_a a la timeline del .spa

In [9]:
dsa_fa_unilat_spa = fun_dsa_b.ajustar_dsa_a_timeline_spa(
    tiempo_dsa=tiempo_fa_unilat,
    dsa=dsa_unilat,
    timeline_spa=timeline_spa_unilat,
    nombre="f_a unilateral",
    verbose=True
)

f_a unilateral: ya coincide con la timeline del .spa. No se modifica.


### Fusión .f_a y .spa

In [10]:
df_merge_fa = fun_dsa.alinear_spa_con_tiempo(
    timeline_spa_unilat,
    df_spa_unilat,
    resolver_duplicados="last"
)

sef_hor = df_merge_fa["SEF08"]
mf_hor = df_merge_fa["MEDFRQ08"]

In [11]:
print("timeline_spa:", len(timeline_spa_unilat))
print("df_merge_fa:", df_merge_fa.shape)

timeline_spa: 2119
df_merge_fa: (2119, 8)


### Cabecera

In [12]:
num_canales, fs, pendiente, offset = fau.extraer_parametros_eeg(ruta_ha_unilat)
print("Parámetros extraídos con éxito:")
print(f" - Canales: {num_canales}")
print(f" - Frecuencia (Hz): {fs}")
print(f" - Pendiente (m): {pendiente:.8f}")
print(f" - Offset (b): {offset:.4f}")

Parámetros extraídos con éxito:
 - Canales: 2
 - Frecuencia (Hz): 128
 - Pendiente (m): 0.05000000
 - Offset (b): -3234.0000


## 1. 3 Archivo ondas crudas .r2a

In [13]:
df_eeg_unilat = fun_dsa_u.leer_r2a(
    archivo_r2a,
    pendiente,
    offset,
    fs=fs
)

In [14]:
df_eeg_recortado, timeline_spa_unilat, info_alineacion = (
    fun_dsa_b.recortar_raw_segun_ta_y_spa(
        df_raw=df_eeg_unilat,
        ruta_ta=ruta_ta_unilat,
        df_spa=df_spa_unilat,
        columna_time="Time",
        fs=128,
        resolver_duplicados="last",
        verbose=True
    )
)

# Usar el inicio del .spa como inicio de la DSA reconstruida
hora_inicio = timeline_spa_unilat.iloc[0]

print("Raw unilateral original:", df_eeg_unilat.shape)
print("Raw unilateral recortado:", df_eeg_recortado.shape)
print("Timeline SPA:", len(timeline_spa_unilat))

=== Timeline .spa ===
Inicio .spa: 2026-03-04 10:35:21
Fin .spa: 2026-03-04 11:10:39
N segundos .spa: 2119


=== Alineación raw a timeline .spa ===
Inicio raw (.t_a): 2026-03-04 10:35:21
Inicio .spa: 2026-03-04 10:35:21
Fin .spa: 2026-03-04 11:10:39
Desfase spa - raw: 0.0 s
Acción inicio: sin_recorte_inicio
Segundos objetivo .spa: 2119
Muestras objetivo: 271232
Muestras raw originales: 271312
Muestras recortadas inicio: 0
Muestras NaN inicio: 0
Muestras recortadas final: 0
Muestras NaN final: 0
Muestras raw alineado: 271232
Duración raw alineado: 2119.0 s
Raw unilateral original: (271312, 5)
Raw unilateral recortado: (271232, 5)
Timeline SPA: 2119


### 1. 3. 1. Reconstrucción

In [15]:
ventana_seg =2
paso_seg =1

df_dsa_canal1, frecuencias_c1 = fun_dsa.crear_matriz_dsa_fft_welch_desde_eeg(
    df_eeg_recortado,
    "canal_1_uV",
    fs=128,
    ventana_seg=ventana_seg,
    paso_seg=paso_seg,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5,
    modo="densidad",
    tiempo_referencia="centro"
)

df_dsa_canal2, _ = fun_dsa.crear_matriz_dsa_fft_welch_desde_eeg(
    df_eeg_recortado,
    "canal_2_uV",
    fs=128,
    ventana_seg=ventana_seg,
    paso_seg=paso_seg,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5,
    modo="densidad", 
    tiempo_referencia="centro"
)

In [16]:
# -------------------- Depende del modo que pongamos las uds son uV^2 o uV^2/Hz (uds. potencia)--------------------

# columnas del dF generado después de la conversión por FFT -> las frecuencias
cols_freq = [c for c in df_dsa_canal1.columns if c != "tiempo_s"]

# copia del dF del canal 1 para tener esa estructura
df_dsa_media = df_dsa_canal1.copy()

# media (en float) de las potencias (en uV) de los 2 canales
# en función de las frecuencias 
pot_media = (
    df_dsa_canal1[cols_freq].to_numpy(dtype=float) +
    df_dsa_canal2[cols_freq].to_numpy(dtype=float)
) / 2

In [17]:

# ----------- CONVERSIÓN A DECIBELIOS (ref documentación del BIS) -----------------------------------
ref_potencia = 0.0001


df_dsa_media[cols_freq] = 10 * np.log10(
    ((df_dsa_media[cols_freq] * 0.5) + 1e-12) / (ref_potencia ** 2)
)

### 1. 3. 2. Adaptación temporal, máscara y plot de DSA EEG

In [18]:
hora_inicio = timeline_spa_unilat.iloc[0]

# ----------------- Adaptación temporal inicial de la DSA reconstruida ---------------------------

tiempo_eeg_tmp, dsa_eeg_tmp = fun_dsa.adaptar_dsa_reconstruida_para_plot(
    df_dsa=df_dsa_media,
    frecuencias=frecuencias_c1,
    hora_inicio=hora_inicio,
    insertar_fila_inicial_nan=True
)

# ---------------- Ajustar reconstruida a la timeline oficial del .spa ---------------------------

dsa_eeg_unilat_spa = fun_dsa_b.ajustar_dsa_a_timeline_spa(
    tiempo_dsa=tiempo_eeg_tmp,
    dsa=dsa_eeg_tmp,
    timeline_spa=timeline_spa_unilat,
    nombre="DSA EEG reconstruida unilateral",
    verbose=True
)

DSA EEG reconstruida unilateral: ya coincide con la timeline del .spa. No se modifica.


In [19]:
# Desde aquí, el tiempo oficial de la reconstruida también es el del .spa
tiempo_eeg_unilat = timeline_spa_unilat.copy()


# Merge del .spa para la reconstruida
df_merge_eeg = fun_dsa.alinear_spa_con_tiempo(
    tiempo=tiempo_eeg_unilat,
    df_spa=df_spa_unilat,
    resolver_duplicados="last"
)

sef_hor = df_merge_eeg["SEF08"]
mf_hor = df_merge_eeg["MEDFRQ08"]

### Máscaras de calidad bilateral - común

In [20]:
_, mask_comun_unilat = fun_dsa.preparar_dsa_con_mask(
    tiempo=timeline_spa_unilat,
    dsa=dsa_eeg_unilat_spa,
    df_merge=df_merge_eeg,
    umbral_sqi=15,
    umbral_ceros=0.9,
    incluir_filas_nan=True
)

mask_comun = mask_comun_unilat.copy()

# en las reconstrucciones se utilizan como valores mínimos y máximos 
# los percentiles más ajustados para replicar el color

print(f"Filas totales:", len(mask_comun))
print(f"\nFilas enmascaradas:", mask_comun.sum())
print(f"\nFilas válidas:", (~mask_comun).sum())

Filas totales: 2119

Filas enmascaradas: 146

Filas válidas: 1973


#### Añadir posibles huecos del .f_a, si existe archivo espectral

In [21]:
# Filas completamente NaN del .f_a ajustado al .spa (comprobación adicional por si acaso)
mask_fa_nan = dsa_fa_unilat_spa.isna().all(axis=1)

mask_comun = mask_comun | mask_fa_nan


print("NaN f_a:", mask_fa_nan.sum())
print("\nFilas enmascaradas finales:", mask_comun.sum())
print("\nFilas válidas finales:", (~mask_comun).sum())

NaN f_a: 0

Filas enmascaradas finales: 146

Filas válidas finales: 1973


#### Aplicación

In [22]:
# DSA reconstruida directa con máscara común ------------------- EEG --------------------------------------------------

""" 
Copia de la dsa proveniente del eeg para incluirla en el plot
 - .loc[mask_comun.values, :]: selecciona todas las filas donde mask_comun vale True, y todas las columnas de frecuencia.
 - np.nan: como el colormap pinta los NaN en blanco, esas filas aparecerán como bandas blancas.
"""

dsa_eeg_plot = dsa_eeg_unilat_spa.copy()
dsa_eeg_plot.loc[mask_comun.values, :] = np.nan

In [23]:
# DSA original f_a con máscara común ------------------------------- FA ---------------------------------------------------

""" 
Copia de la dsa proveniente del f_a para incluirla en el plot
 - .loc[mask_comun.values, :]: selecciona todas las filas donde mask_comun vale True, y todas las columnas de frecuencia.
 - np.nan: como el colormap pinta los NaN en blanco, esas filas aparecerán como bandas blancas.
"""

dsa_fa_plot = dsa_fa_unilat_spa.copy()
dsa_fa_plot.loc[mask_comun.values, :] = np.nan

#### Comprobación

In [24]:
# ============================================================
# Comprobación de bandas blancas tras aplicar máscara unilateral
# ============================================================

# 1. Detectar filas completamente blancas en cada matriz
mask_blanca_eeg = dsa_eeg_plot.isna().all(axis=1)
mask_blanca_fa = dsa_fa_plot.isna().all(axis=1)


# 2. Comprobar que las bandas blancas coinciden con la máscara aplicada
print("=== COMPROBACIÓN UNILATERAL ===")

print("EEG coincide con mask_comun:")
print((mask_blanca_eeg.values == mask_comun.values).all())

print("FA coincide con mask_comun:")
print((mask_blanca_fa.values == mask_comun.values).all())

print("\nBandas blancas EEG:", mask_blanca_eeg.sum())
print("Bandas blancas FA:", mask_blanca_fa.sum())
print("Filas en mask_comun:", mask_comun.sum())

print("\nDiferencias EEG vs máscara:")
print((mask_blanca_eeg.values != mask_comun.values).sum())

print("Diferencias FA vs máscara:")
print((mask_blanca_fa.values != mask_comun.values).sum())


# 3. Tabla para localizar posibles diferencias
df_check_blancas_unilat = pd.DataFrame({
    "Time": timeline_spa_unilat.reset_index(drop=True),

    "mask_comun": mask_comun.reset_index(drop=True),
    "blanca_eeg": mask_blanca_eeg.reset_index(drop=True),
    "blanca_fa": mask_blanca_fa.reset_index(drop=True),
})

df_check_blancas_unilat["diff_eeg"] = (
    df_check_blancas_unilat["mask_comun"] !=
    df_check_blancas_unilat["blanca_eeg"]
)

df_check_blancas_unilat["diff_fa"] = (
    df_check_blancas_unilat["mask_comun"] !=
    df_check_blancas_unilat["blanca_fa"]
)


# 4. Mostrar solo filas problemáticas, si las hay
df_diferencias_blancas_unilat = df_check_blancas_unilat[
    df_check_blancas_unilat[["diff_eeg", "diff_fa"]].any(axis=1)
]

print("\nNúmero total de filas con alguna diferencia:")
print(len(df_diferencias_blancas_unilat))

display(df_diferencias_blancas_unilat.head(50))

=== COMPROBACIÓN UNILATERAL ===
EEG coincide con mask_comun:
True
FA coincide con mask_comun:
True

Bandas blancas EEG: 146
Bandas blancas FA: 146
Filas en mask_comun: 146

Diferencias EEG vs máscara:
0
Diferencias FA vs máscara:
0

Número total de filas con alguna diferencia:
0


,Time,mask_comun,blanca_eeg,blanca_fa,diff_eeg,diff_fa


### 1. 3. 3. Preparar escala de color de f_a y eeg reconstruida

In [25]:
# DSA reconstruida ------------------------------------------------ EEG -----------------------------------------------

matriz_eeg, vmin_eeg, vmax_eeg, norm_eeg, cmap_eeg = fun_dsa.preparar_escala_color_dsa(
    dsa_eeg_plot,
    gamma=0.4
)

In [26]:
# DSA original ---------------------------------------------------- FA --------------------------------------------------

# las matrices que vienen de la f_a suelen mostrar valores entre el 49 y 94
matriz_fa, vmin_fa, vmax_fa, norm_fa, cmap_fa = fun_dsa.preparar_escala_color_dsa(
    dsa_fa_plot,
    vmin=49,
    vmax=94,
    gamma=1
)

### 1. 3. 4. Visualización r2a

In [27]:
# Celda de visualizacion omitida en esta copia ejecutable para anexos.

###  Visualización .f_a

In [28]:
# Celda de visualizacion omitida en esta copia ejecutable para anexos.

## 1. 4. Comprobaciones

In [29]:
# ============================================================
# Comprobación de tamaños unilateral
# ============================================================

print("timeline_spa_unilat:", len(timeline_spa_unilat))

print("FA ajustado:", dsa_fa_unilat_spa.shape)
print("EEG ajustado:", dsa_eeg_unilat_spa.shape)

print("merge FA:", df_merge_fa.shape)
print("merge EEG:", df_merge_eeg.shape)

print("plot FA:", dsa_fa_plot.shape)
print("plot EEG:", dsa_eeg_plot.shape)

print("mask_comun:", len(mask_comun))

timeline_spa_unilat: 2119
FA ajustado: (2119, 60)
EEG ajustado: (2119, 60)
merge FA: (2119, 8)
merge EEG: (2119, 8)
plot FA: (2119, 60)
plot EEG: (2119, 60)
mask_comun: 2119


In [30]:
# ============================================================
# Rango de valores - DSA reconstruida
# ============================================================

cols_freq_eeg = [c for c in dsa_eeg_plot.columns]

print("Mínimos y máximos - RECONSTRUCCIÓN")
print(np.nanmin(dsa_eeg_plot[cols_freq_eeg].values))
print(np.nanmax(dsa_eeg_plot[cols_freq_eeg].values))

print("\nPercentiles mínimos y máximos - RECONSTRUCCIÓN")
print(np.nanpercentile(dsa_eeg_plot[cols_freq_eeg].values, 2))
print(np.nanpercentile(dsa_eeg_plot[cols_freq_eeg].values, 99.5))

Mínimos y máximos - RECONSTRUCCIÓN
18.20106203150211
131.8024781807748

Percentiles mínimos y máximos - RECONSTRUCCIÓN
64.25061065476443
115.87254232412026


In [31]:
# ============================================================
# Rango de valores - DSA .f_a
# ============================================================

cols_freq_fa = [c for c in dsa_fa_plot.columns]

print("Mínimos y máximos - FA")
print(np.nanmin(dsa_fa_plot[cols_freq_fa].values))
print(np.nanmax(dsa_fa_plot[cols_freq_fa].values))

print("\nPercentiles mínimos y máximos - FA")
print(np.nanpercentile(dsa_fa_plot[cols_freq_fa].values, 2))
print(np.nanpercentile(dsa_fa_plot[cols_freq_fa].values, 99.5))

Mínimos y máximos - FA
55.61
103.02

Percentiles mínimos y máximos - FA
70.0
96.87


## 1. 5. Métricas y comparación

### 1. 5. 1.  Comparación base

In [32]:
dsa_eeg_comparacion, dsa_fa_comparacion = fun_dsa.preparar_matrices_para_comparacion(
    dsa_eeg_plot,
    dsa_fa_plot
)

dsa_eeg_z = fun_dsa.zscore_global(dsa_eeg_comparacion)
dsa_fa_z = fun_dsa.zscore_global(dsa_fa_comparacion)

metricas_base = fun_dsa.comparar_dsa_global(
    dsa_eeg_z,
    dsa_fa_z
)

metricas_base

{'n_valores_comparados': 118380,
 'MAE': 0.838223498969729,
 'RMSE': 1.0870007115405225,
 'bias_B_menos_A': -9.555533327674265e-16,
 'Pearson': 0.4092147265551989,
 'Spearman': 0.33303967731108264}

## 1. 6. Suavizado + shift

In [33]:
dsa_eeg_comp_sin_mask, dsa_fa_comp_sin_mask = fun_dsa.preparar_matrices_para_comparacion(
    dsa_eeg_unilat_spa,
    dsa_fa_unilat_spa
)

In [34]:
ventanas_sp_smooth_probar = [1, 5, 10, 30, 60]

print("Ventanas a probar:", ventanas_sp_smooth_probar)
print("Ventana indicada por SpSmooth:", suavizado_spsmooth_unilat, "s")

ventanas_sp_smooth_unilat = [suavizado_spsmooth_unilat]

Ventanas a probar: [1, 5, 10, 30, 60]
Ventana indicada por SpSmooth: 30 s


In [35]:
df_suav_shift = fun_dsa.probar_suavizado_y_shifts(
    dsa_eeg_comp_sin_mask,
    dsa_fa_comp_sin_mask,
    ventanas_suavizado= ventanas_sp_smooth_probar,
    shifts=range(0, 31)
)

df_suav_shift.sort_values("Pearson", ascending=False).head(7)

,suavizado_s,shift_s,Pearson,Spearman,MAE,RMSE
102,30,9,0.663404,0.646895,0.609884,0.820293
103,30,10,0.663241,0.646206,0.610244,0.820502
101,30,8,0.662953,0.646635,0.610258,0.820835
104,30,11,0.662464,0.644579,0.611381,0.821458
100,30,7,0.661905,0.645435,0.611360,0.822105
105,30,12,0.661085,0.642046,0.613265,0.823146
99,30,6,0.660289,0.643329,0.613141,0.824064


### 1. 6. 1. Tabla resumen final

In [36]:
mejor_pearson = df_suav_shift.sort_values("Pearson", ascending=False).iloc[0]
mejor_spearman = df_suav_shift.sort_values("Spearman", ascending=False).iloc[0]

df_resumen_final = pd.DataFrame([
    {
        "criterio": "Mejor Pearson",
        "suavizado_s": mejor_pearson["suavizado_s"],
        "shift_s": mejor_pearson["shift_s"],
        "Pearson": mejor_pearson["Pearson"],
        "Spearman": mejor_pearson["Spearman"],
        "MAE": mejor_pearson["MAE"],
        "RMSE": mejor_pearson["RMSE"],
    },
    {
        "criterio": "Mejor Spearman",
        "suavizado_s": mejor_spearman["suavizado_s"],
        "shift_s": mejor_spearman["shift_s"],
        "Pearson": mejor_spearman["Pearson"],
        "Spearman": mejor_spearman["Spearman"],
        "MAE": mejor_spearman["MAE"],
        "RMSE": mejor_spearman["RMSE"],
    }
])

df_resumen_final

,criterio,suavizado_s,shift_s,Pearson,Spearman,MAE,RMSE
0,Mejor Pearson,30.0,9.0,0.663404,0.646895,0.609884,0.820293
1,Mejor Spearman,30.0,9.0,0.663404,0.646895,0.609884,0.820293


## 1. 7. Optimizaciones - Suavizado limpio

In [37]:
suavizado_final = suavizado_spsmooth_unilat

dsa_eeg_suav_unilat = dsa_eeg_unilat_spa.copy().rolling(
    window=suavizado_final,
    min_periods=1,
    center=False
).mean()

In [38]:
# máscara común al final
dsa_eeg_suav_plot_unilat = dsa_eeg_suav_unilat.copy()
dsa_eeg_suav_plot_unilat.loc[mask_comun.values, :] = np.nan

matriz_opt_unilat, vmin_opt_unilat, vmax_opt_unilat, norm_opt_unilat, cmap_opt_unilat = fun_dsa.preparar_escala_color_dsa(
    dsa_eeg_suav_plot_unilat,
    gamma=0.25
)

### 1. 7. 1. Comprobación bandas blancas

In [39]:
mask_blanca_suav = dsa_eeg_suav_plot_unilat.isna().all(axis=1)
mask_blanca_eeg = dsa_eeg_plot.isna().all(axis=1)
mask_blanca_fa = dsa_fa_plot.isna().all(axis=1)

print("bandas blancas directa vs suavizada")
print((mask_blanca_eeg == mask_blanca_suav).all())

print("Diferencias:")
print((mask_blanca_eeg != mask_blanca_suav).sum())

bandas blancas directa vs suavizada
True
Diferencias:
0


## 1. 8. Optimizaciones - Suavizado + shift

In [40]:
#shift_final_opcion = int(mejor_pearson["shift_s"])  # en tu caso 10

shift_final = SHIFT_UNILAT

# matriz desplazada sin máscara
dsa_eeg_suav_shift_full = pd.DataFrame(
    np.nan,
    index=dsa_eeg_suav_unilat.index,
    columns=dsa_eeg_suav_unilat.columns
)

dsa_eeg_suav_shift_full.iloc[shift_final:, :] = (
    dsa_eeg_suav_unilat.iloc[:-shift_final, :].to_numpy()
)

# matriz desplazada para plot, con máscara
dsa_eeg_suav_shift_plot = dsa_eeg_suav_shift_full.copy()
dsa_eeg_suav_shift_plot.loc[mask_comun.values, :] = np.nan


In [41]:
matriz_opt_desfase_full, vmin_opt_desfase_full, vmax_opt_desfase_full, norm_opt_desfase_full, cmap_opt_desfase_full = fun_dsa.preparar_escala_color_dsa(
    dsa_eeg_suav_shift_plot,
    gamma=0.25
)

print("Tiempo:", len(timeline_spa_unilat))
#print("f_a:", matriz_fa.shape)
print("shift full:", matriz_opt_desfase_full.shape)

Tiempo: 2119
shift full: (2119, 60)


In [42]:

from IPython.display import display, Markdown

display(Markdown("## Resumen final ejecutado"))

try:
    df_resumen_unilat = pd.DataFrame([{
        "registro": "L03041035",
        "modo": "unilateral",
        "suavizado_s": int(mejor_pearson["suavizado_s"]),
        "shift_s": int(mejor_pearson["shift_s"]),
        "Pearson": float(mejor_pearson["Pearson"]),
        "Spearman": float(mejor_pearson["Spearman"]),
        "MAE": float(mejor_pearson["MAE"]),
        "RMSE": float(mejor_pearson["RMSE"]),
    }])
    display(df_resumen_unilat)
    display(Markdown("### Top 10 combinaciones de suavizado y shift"))
    display(df_suav_shift.sort_values("Pearson", ascending=False).head(10))
except Exception as exc:
    display(Markdown(f"No se pudo generar el resumen unilateral: `{exc}`"))


## Resumen final ejecutado

,registro,modo,suavizado_s,shift_s,Pearson,Spearman,MAE,RMSE
0,L03041035,unilateral,30,9,0.663404,0.646895,0.609884,0.820293


### Top 10 combinaciones de suavizado y shift

,suavizado_s,shift_s,Pearson,Spearman,MAE,RMSE
102,30,9,0.663404,0.646895,0.609884,0.820293
103,30,10,0.663241,0.646206,0.610244,0.820502
101,30,8,0.662953,0.646635,0.610258,0.820835
104,30,11,0.662464,0.644579,0.611381,0.821458
100,30,7,0.661905,0.645435,0.611360,0.822105
105,30,12,0.661085,0.642046,0.613265,0.823146
99,30,6,0.660289,0.643329,0.613141,0.824064
106,30,13,0.659123,0.638650,0.615857,0.825536
98,30,5,0.658136,0.640379,0.615512,0.826669
107,30,14,0.656601,0.634452,0.619078,0.828594
